# technical correction and post hoc exploratory optimization

This notebook has two independent
workstreams:

- **Workstream A** re-runs the frozen model-training and primary-evaluation pipeline once, using a corrected elastic-net
  tuning failure policy (`correction-spec.md`), and reconciles every cell against the archived
  primary analysis. The grid, splits, endpoints, horizons, metrics, precision targets, materiality
  margins, and classification rule are byte-identical to the frozen originals.
- **Workstream B** is a **post hoc, prospectively specified exploratory model-development analysis**
  (`optimization-spec.md`) testing whether a closed set of five additional delayed-entry-compatible
  survival models (`B2`-`B6`) can improve on the frozen `minimal_cox` benchmark (`B1`, loaded from
  the frozen registry and never refit) for the primary 730-day OS target, with TTNT-D supportive
  only.

**Every Workstream B number in this notebook carries `analysis_status = post_hoc_exploratory`.**
Because the decision to perform this optimization was informed by the frozen primary-evaluation results, Workstream B's
new held-out estimates are useful exploratory evidence, **not independent external confirmation**.
The frozen primary-evaluation bundle remains the project's primary analysis and is verified byte-for-byte
unchanged, before and after, throughout this notebook.

**Scope boundary for this correction and exploratory optimization:** no explanation, attribution, permutation importance, SHAP, prediction-
stability, sensitivity, calibration-drift, sequencing-lag, decision-curve, net-benefit, or
final-all-centre-refit code path exists anywhere in this notebook. This notebook contains no
handwritten fitting, metric, or bootstrap logic and displays no patient-level row, key, time,
prediction, or coefficient -- every number below comes from a call to a tested orchestration
boundary in `crc_folfox.optimize`.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS_DIRECTORY = PROJECT_ROOT / "analysis" / "out" / "current"
FEATURE_MANIFEST_PATH = PROJECT_ROOT / "src" / "crc_folfox" / "features" / "feature_manifest.json"
ARCHIVE_ROOT = PROJECT_ROOT / "analysis" / "out" / "archive"
OUTPUT_ROOT = PROJECT_ROOT / "analysis" / "out" / "current"
WORKSTREAM_B_DIRECTORY = OUTPUT_ROOT / "phase5a" / "optimization"

## Archived model-training and primary-evaluation bundle

The model-training and primary-evaluation outputs were archived byte-for-byte before any
correction or exploratory optimization code ran
(Gate 1). The same six regression hashes are re-checked here, live, before this notebook runs any
Workstream A or B orchestration -- and are re-checked again at the very end, after every real call
this notebook makes, so any silent drift blocks the notebook rather than passing quietly.


In [2]:
from crc_folfox.optimize.archive import verify_frozen_primary_unchanged

verification_before = verify_frozen_primary_unchanged(ANALYSIS_DIRECTORY, FEATURE_MANIFEST_PATH)
display(verification_before)

FrozenPrimaryVerification(cleaning_run_manifest_sha256='d11982f0e9cd7f740cae7432744fa23aba336d944f7236a2f10429e775b84bfd', analysis_dataset_sha256='6b9e28099f547150324b33be900d228d22b411b7dbe0d68e1e08551aaab6740e', feature_manifest_sha256='cc891bcdca2b06179dd33e4bc55160e0e048557059e57b0b02789c7b18dc0f73', model_training_fit_summary_sha256='dfb77a84220741ad94e1362a90eaaa8c2240980ddd7c59ef0ed039548d60ad9b', model_training_registry_sha256='2695e55ae3ed97a9e98d5ff419544dafb7ee21e9bb65c6e6ca4ae6bd3770596f', primary_evaluation_manifest_sha256='1e25224a40ce91b4f88d16410f658291f2440e83209e4330a70d274a2860ba15')

## Workstream A -- the elastic-net tuning failure-policy correction

`model-training-spec.md`'s per-candidate elastic-net tuning loop caught `Exception` broadly around
both the fit and score calls, which meant one candidate's convergence failure could silently
disqualify sibling candidates in the same inner fold rather than only that one candidate --
`correction-spec.md`'s 2026-08-09 amendment. The corrected policy in
`crc_folfox.model_training.selection` isolates failure to exactly the failing
`(penalizer, l1_ratio, inner fold)` combination, using an explicit `EXPECTED_TUNING_EXCEPTIONS`
tuple with no bare `except Exception` anywhere on the tuning-candidate path.

The cell below re-runs the complete frozen model-training -> primary-evaluation pipeline once, with only this fix
applied, into a separate correction output directory, and reconciles
every cell of the corrected run against the archived primary analysis. **The frozen directories are
never written to.**


In [3]:
from crc_folfox.optimize.correction import run_workstream_a_correction

correction_result = run_workstream_a_correction(
    ANALYSIS_DIRECTORY, FEATURE_MANIFEST_PATH, ARCHIVE_ROOT, OUTPUT_ROOT
)
display(correction_result.movement_summary)

/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/src/crc_folfox/evaluate/pooling.py:161: RuntimeWarning: Mean of empty slice
  conditional_mean = np.nanmean(np.stack(per_repeat_conditional), axis=0)


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/src/crc_folfox/evaluate/pooling.py:161: RuntimeWarning: Mean of empty slice
  conditional_mean = np.nanmean(np.stack(per_repeat_conditional), axis=0)


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/src/crc_folfox/evaluate/pooling.py:161: RuntimeWarning: Mean of empty slice
  conditional_mean = np.nanmean(np.stack(per_repeat_conditional), axis=0)


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/src/crc_folfox/evaluate/pooling.py:161: RuntimeWarning: Mean of empty slice
  conditional_mean = np.nanmean(np.stack(per_repeat_conditional), axis=0)


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/src/crc_folfox/evaluate/pooling.py:161: RuntimeWarning: Mean of empty slice
  conditional_mean = np.nanmean(np.stack(per_repeat_conditional), axis=0)


...

/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/links.py:989: RuntimeWarning: overflow encountered in exp
  return 1 - np.exp(-np.exp(z))


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/links.py:989: RuntimeWarning: overflow encountered in exp
  return 1 - np.exp(-np.exp(z))


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)


/Users/camilasarmiento/Documents/Data Freelancer/portfolio/CRC/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/links.py:989: RuntimeWarning: overflow encountered in exp
  return 1 - np.exp(-np.exp(z))


{'unchanged': 57, 'recovered': 70, 'still_failed': 0, 'shifted': 41}

### Reconciliation table

Every cell of the archived `performance_summary.csv` is classified into exactly one of five
movement classes: `unchanged`, `recovered` (`not_computable` -> a computable status),
`still_failed`, `shifted` (an `elastic_net_cox` `pooled` cell whose status didn't move but whose
point estimate did, because recovering elsewhere changed that cell's admissible pooled-repeat
composition -- `correction-spec.md`'s 2026-08-09 amendment), or `unexpected` (any other change,
which blocks publication -- `assert_no_unexpected_movement` already ran inside the cell above and
would have raised before reaching here).


In [4]:
display(
    correction_result.reconciliation[
        [
            "endpoint",
            "design",
            "centre",
            "model",
            "metric",
            "horizon",
            "movement_class",
            "reason_archived",
            "reason_corrected",
        ]
    ].head(20)
)

,endpoint,design,centre,model,metric,horizon,movement_class,reason_archived,reason_corrected
0,OS,iecv,DFCI,minimal_cox,concordance,<NA>,unchanged,NaN,
1,OS,iecv,DFCI,minimal_cox,ibs,<NA>,unchanged,NaN,
2,OS,iecv,DFCI,minimal_cox,auc,365.0,unchanged,insufficient_events,insufficient_events
3,OS,iecv,DFCI,minimal_cox,brier,365.0,unchanged,insufficient_events,insufficient_events
4,OS,iecv,DFCI,minimal_cox,slope,365.0,unchanged,"insufficient_events,precision_target_missed","insufficient_events,precision_target_missed"
5,OS,iecv,DFCI,minimal_cox,citl,365.0,unchanged,"insufficient_events,precision_target_missed","insufficient_events,precision_target_missed"
6,OS,iecv,DFCI,minimal_cox,slope,365.0,unchanged,"insufficient_events,precision_target_missed","insufficient_events,precision_target_missed"
7,OS,iecv,DFCI,minimal_cox,citl,365.0,unchanged,"insufficient_events,precision_target_missed","insufficient_events,precision_target_missed"
8,OS,iecv,DFCI,minimal_cox,auc,730.0,unchanged,NaN,
9,OS,iecv,DFCI,minimal_cox,brier,730.0,unchanged,NaN,


## Workstream B candidate register

The candidate list is closed (`optimization-spec.md`, "Candidate model set"). `B1` is the frozen
benchmark, never refit. `B2`-`B6` are the five new candidates this pass evaluates. `B7`/`B8` are
conditional and were not admitted.


In [ ]:
candidate_register = pd.DataFrame(
    [
        {
            "candidate": "B1 minimal_cox",
            "software": "lifelines (frozen model-training bundle, not refit)",
            "predictor_rule": "4-variable minimal Cox",
            "delayed_entry_evidence": "model-training delayed-entry fixtures (unchanged)",
            "status": "benchmark, loaded from frozen registry",
        },
        {
            "candidate": "B2 glmnet_ridge_core",
            "software": "R glmnet (Surv(entry, exit, event, type='counting'), family='cox')",
            "predictor_rule": "preprocessing core matrix, ridge (alpha=0)",
            "delayed_entry_evidence": "test_r_adapter.py: entry-perturbation, entry-zero, fixtures",
            "status": "admitted, evaluated",
        },
        {
            "candidate": "B3 glmnet_enet_core",
            "software": "R glmnet, alpha in {0.5, 1.0}",
            "predictor_rule": "preprocessing core matrix, tuned elastic net",
            "delayed_entry_evidence": "test_r_adapter.py: entry-perturbation, entry-zero, fixtures",
            "status": "admitted, evaluated",
        },
        {
            "candidate": "B4 penalized_nonlinear_cox",
            "software": "lifelines, ridge (l1_ratio=0.0)",
            "predictor_rule": "core + age spline, + presentation x age-spline interaction",
            "delayed_entry_evidence": "shared model-training delayed-entry Cox machinery (unchanged)",
            "status": "admitted, evaluated",
        },
        {
            "candidate": "B5 reduced_core_penalized_cox",
            "software": "lifelines, ridge (l1_ratio=0.0)",
            "predictor_rule": "core matrix, stricter category reduction (25 obs / 10 events)",
            "delayed_entry_evidence": "shared model-training delayed-entry Cox machinery (unchanged)",
            "status": "admitted, evaluated",
        },
        {
            "candidate": "B6 flexsurv_rp_spline",
            "software": "R flexsurv (Surv(entry, exit, event, type='counting'), flexsurvspline)",
            "predictor_rule": "4 minimal-Cox variables, k in {1, 2} internal knots",
            "delayed_entry_evidence": "test_r_adapter.py: entry-perturbation, entry-zero, fixtures",
            "status": "admitted, evaluated",
        },
        {
            "candidate": "B7 ltrc_rrf",
            "software": "LTRCforests::ltrcrrf",
            "predictor_rule": "n/a",
            "delayed_entry_evidence": "n/a",
            "status": "technically_ineligible (model-training segfault on fixture); retry not attempted",
        },
        {
            "candidate": "B8 ltrc_boosting",
            "software": "n/a",
            "predictor_rule": "n/a",
            "delayed_entry_evidence": "n/a",
            "status": "technically_ineligible (no maintained implementation identified)",
        },
    ]
)
display(candidate_register)

,candidate,software,predictor_rule,delayed_entry_evidence,status
0,B1 minimal_cox,"lifelines (frozen model-training bundle, not refit)",4-variable minimal Cox,model-training delayed-entry fixtures (unchanged),"benchmark, loaded from frozen registry"
1,B2 glmnet_ridge_core,"R glmnet (Surv(entry, exit, event, type='count...","preprocessing core matrix, ridge (alpha=0)","test_r_adapter.py: entry-perturbation, entry-z...","admitted, evaluated"
2,B3 glmnet_enet_core,"R glmnet, alpha in {0.5, 1.0}","preprocessing core matrix, tuned elastic net","test_r_adapter.py: entry-perturbation, entry-z...","admitted, evaluated"
3,B4 penalized_nonlinear_cox,"lifelines, ridge (l1_ratio=0.0)","core + age spline, + presentation x age-spline...",shared model-training delayed-entry Cox machinery (un...,"admitted, evaluated"
4,B5 reduced_core_penalized_cox,"lifelines, ridge (l1_ratio=0.0)","core matrix, stricter category reduction (25 o...",shared model-training delayed-entry Cox machinery (un...,"admitted, evaluated"
5,B6 flexsurv_rp_spline,"R flexsurv (Surv(entry, exit, event, type='cou...","4 minimal-Cox variables, k in {1, 2} internal ...","test_r_adapter.py: entry-perturbation, entry-z...","admitted, evaluated"
6,B7 ltrc_rrf,LTRCforests::ltrcrrf,n/a,n/a,technically_ineligible (model-training segfault on de...
7,B8 ltrc_boosting,n/a,n/a,n/a,technically_ineligible (no maintained implemen...


## Nested IECV design, entry support, and the development-fitting scope inventory

Each outer institution is held out in turn; all category reduction, feature construction, tuning,
and configuration selection happen entirely within the other two institutions (the preprocessing inner
splits), and the selected configuration is applied once to the held-out institution. `minimal_cox`
is never refit. Entry support is computed once per endpoint directly from the verified assessment
frame -- it depends only on entry/exit/event, not on any candidate, so it is identical for
Workstream A, B, and the frozen primary evaluation by construction.


In [6]:
from crc_folfox.optimize.evaluation import workstream_b_entry_support_summary

entry_support = workstream_b_entry_support_summary(ANALYSIS_DIRECTORY, endpoints=("OS", "TTNT-D"))
display(entry_support)

,endpoint,group,day,n_eligible,n_events,n_event_free,n_censored_before
0,OS,DFCI,84.0,86,1,83,2
1,OS,DFCI,365.0,150,16,128,6
2,OS,DFCI,730.0,184,51,119,14
3,OS,MSK,84.0,136,3,133,0
4,OS,MSK,365.0,214,26,183,5
5,OS,MSK,730.0,269,77,178,14
6,OS,VICC,84.0,15,0,14,1
7,OS,VICC,365.0,41,6,31,4
8,OS,VICC,730.0,63,21,33,9
9,OS,All,84.0,237,4,230,3


The cell below fits `B2`-`B6` across every development scope for both endpoints (56 scopes: 53 OS
-- 3 IECV outer + 50 pooled -- and 3 TTNT-D IECV-outer-only; the pooled comparator is prospectively
omitted for the supportive endpoint) and publishes the two registries
(`model_registry.json` for the Python-fit `B4`/`B5`, `r_model_registry.json` for the R-fit
`B2`/`B3`/`B6`) plus `fit_summary.csv` in the dedicated exploratory-optimization output directory.


In [7]:
from crc_folfox.optimize.workflow import run_workstream_b_development

development_result = run_workstream_b_development(
    ANALYSIS_DIRECTORY, WORKSTREAM_B_DIRECTORY, endpoints=("OS", "TTNT-D")
)
fit_status_inventory = (
    development_result.fit_summary.groupby(["candidate", "fit_status"], sort=True)
    .agg(rows=("model_id", "count"))
    .reset_index()
)
display(fit_status_inventory)

,candidate,fit_status,rows
0,flexsurv_rp_spline,fit_failed,32
1,flexsurv_rp_spline,fit_succeeded,24
2,glmnet_enet_core,fit_succeeded,56
3,glmnet_ridge_core,fit_succeeded,56
4,penalized_nonlinear_cox,fit_succeeded,56
5,reduced_core_penalized_cox,fit_succeeded,56


## Discrimination, calibration, and prediction-error tables

`run_workstream_b_iecv_evaluation` reuses the primary-evaluation workflow's own `_score_cell`/`_complexity_rows` boundary
verbatim -- the same metric set, the same paired bootstrap resamples (2000, with index matrices
proven identical to the primary evaluation's by a direct identity test), and the same
reporting-status rule. **This pass covers the IECV design only** for both
endpoints -- the pooled-design transportability comparison for `B2`-`B6` is prospectively deferred
(`optimization-spec.md`'s 2026-08-09 amendment).


In [ ]:
from crc_folfox.optimize.evaluation import run_workstream_b_evaluation

performance_summary, transportability_summary, complexity_value_summary = (
    run_workstream_b_evaluation(
        ANALYSIS_DIRECTORY, WORKSTREAM_B_DIRECTORY, endpoints=("OS", "TTNT-D")
    )
)
performance_summary.to_csv(WORKSTREAM_B_DIRECTORY / "performance_summary.csv", index=False)
transportability_summary.to_csv(
    WORKSTREAM_B_DIRECTORY / "transportability_summary.csv", index=False
)
complexity_value_summary.to_csv(
    WORKSTREAM_B_DIRECTORY / "complexity_value_summary.csv", index=False
)

reporting_status_counts = performance_summary["reporting_status"].value_counts()
display(reporting_status_counts)

reporting_status
estimable_imprecise    395
estimable              263
not_computable          98
Name: count, dtype: int64

In [ ]:
display(
    performance_summary[
        [
            "endpoint",
            "design",
            "centre",
            "model",
            "metric",
            "horizon",
            "risk_scale",
            "estimate",
            "ci_lower",
            "ci_upper",
            "n_eligible",
            "n_events",
            "reporting_status",
        ]
    ].head(30)
)

,endpoint,design,centre,model,metric,horizon,risk_scale,estimate,ci_lower,ci_upper,n_eligible,n_events,reporting_status
0,OS,iecv,DFCI,minimal_cox,concordance,NaN,unconditional,0.668727,0.613807,0.721806,227,129,estimable
1,OS,iecv,DFCI,minimal_cox,ibs,NaN,unconditional,0.167130,0.128916,0.207500,184,51,estimable
2,OS,iecv,DFCI,minimal_cox,auc,365.0,unconditional,0.675138,0.550361,0.799338,150,16,estimable_imprecise
3,OS,iecv,DFCI,minimal_cox,brier,365.0,unconditional,0.097455,0.058781,0.141673,150,16,estimable_imprecise
4,OS,iecv,DFCI,minimal_cox,slope,365.0,unconditional,1.078809,0.413384,2.036985,150,16,estimable_imprecise
5,OS,iecv,DFCI,minimal_cox,citl,365.0,unconditional,0.475061,-0.090806,0.894424,150,16,estimable_imprecise
6,OS,iecv,DFCI,minimal_cox,slope,365.0,entry_conditional,1.356828,0.757481,2.306748,150,16,estimable_imprecise
7,OS,iecv,DFCI,minimal_cox,citl,365.0,entry_conditional,0.683070,0.111430,1.116470,150,16,estimable_imprecise
8,OS,iecv,DFCI,minimal_cox,auc,730.0,unconditional,0.682666,0.599178,0.761335,184,51,estimable
9,OS,iecv,DFCI,minimal_cox,brier,730.0,unconditional,0.202202,0.165339,0.241387,184,51,estimable


## Calibration and transportability

**One disclosed gap remains (`optimization-spec.md`'s 2026-08-09 amendment), not a silent
omission:** no quintile-curve CSV or PNG figure was generated this pass -- the primary evaluation's quintile
calibration-curve computation is coupled to its own atomic-publish machinery in a way not yet
extended to Workstream B. Calibration is still reported in full numerically: every cell above
already carries the calibration-in-the-large (`citl`) and slope (`slope`) point estimates and
intervals on both the unconditional and entry-conditional risk scales, shown side by side below as
a coherence check.

The pooled-design transportability gap (`gap = pooled - iecv`, `optimization-spec.md`'s "Comparison
rules") now covers all six candidates for OS (`optimization-spec.md`'s 2026-08-10 amendment,
generalizing the primary-evaluation workflow's `aggregate_pooled_predictions` -- cross-checked against it directly for
`minimal_cox` and confirmed byte-identical). It is shown in its own section below, right after the
coherence check.


In [ ]:
coherence = (
    performance_summary[performance_summary["metric"].isin(["citl", "slope"])]
    .pivot_table(
        index=["endpoint", "design", "centre", "model", "metric", "horizon"],
        columns="risk_scale",
        values="estimate",
    )
    .reset_index()
)
display(coherence.head(20))

risk_scale,endpoint,design,centre,model,metric,horizon,entry_conditional,unconditional
0,OS,iecv,DFCI,glmnet_enet_core,citl,365.0,-0.125610,-0.389287
1,OS,iecv,DFCI,glmnet_enet_core,citl,730.0,-0.074291,-0.305345
2,OS,iecv,DFCI,glmnet_enet_core,slope,365.0,1.173467,0.876552
3,OS,iecv,DFCI,glmnet_enet_core,slope,730.0,0.946411,1.053243
4,OS,iecv,DFCI,glmnet_ridge_core,citl,365.0,-0.153220,-0.403964
5,OS,iecv,DFCI,glmnet_ridge_core,citl,730.0,-0.083828,-0.319199
6,OS,iecv,DFCI,glmnet_ridge_core,slope,365.0,3.190357,3.123617
7,OS,iecv,DFCI,glmnet_ridge_core,slope,730.0,1.408489,2.622816
8,OS,iecv,DFCI,minimal_cox,citl,365.0,0.683070,0.475061
9,OS,iecv,DFCI,minimal_cox,citl,730.0,0.519889,0.335969


## Transportability gap

`gap = pooled - iecv` on the same eligible OS patients and the same paired bootstrap resamples, for
the ten primary metric-horizon combinations, covering all six candidates. `higher_is_better` is
recorded explicitly per metric so direction is unambiguous. A negative IBS gap means the pooled
(mixed-centre) model has *lower* (better) integrated Brier score than the institution-held-out
model for that centre -- i.e. excluding the centre from training cost some performance; a positive
gap means the institution-held-out model was actually *better*, i.e. this institution's data added
noise more than signal for it specifically.


In [ ]:
display(
    transportability_summary[
        [
            "centre",
            "model",
            "metric",
            "horizon",
            "higher_is_better",
            "gap",
            "gap_ci_lower",
            "gap_ci_upper",
            "pooled_status",
            "iecv_status",
        ]
    ][transportability_summary["metric"] == "ibs"].sort_values(["model", "centre"])
)

,centre,model,higher_is_better,gap,gap_ci_lower,gap_ci_upper,pooled_status,iecv_status
51,DFCI,flexsurv_rp_spline,False,NaN,NaN,NaN,not_computable,not_computable
111,MSK,flexsurv_rp_spline,False,NaN,NaN,NaN,not_computable,estimable
171,VICC,flexsurv_rp_spline,False,NaN,NaN,NaN,not_computable,not_computable
21,DFCI,glmnet_enet_core,False,-0.000918,-0.006395,0.004426,estimable,estimable
81,MSK,glmnet_enet_core,False,-0.010026,-0.016564,-0.003377,estimable,estimable
141,VICC,glmnet_enet_core,False,-0.005883,-0.014168,0.000996,estimable_imprecise,estimable_imprecise
11,DFCI,glmnet_ridge_core,False,0.000755,-0.000637,0.002056,estimable,estimable
71,MSK,glmnet_ridge_core,False,-0.002162,-0.004310,0.000180,estimable,estimable
131,VICC,glmnet_ridge_core,False,-0.001151,-0.004615,0.001844,estimable_imprecise,estimable_imprecise
1,DFCI,minimal_cox,False,0.001072,-0.000610,0.002740,estimable,estimable


## Complexity-value table and success criteria

For each of `B2`-`B6` compared with `minimal_cox` on the same eligible patients and paired
resamples: seven gains and the locked mechanical classification into **clear added value**,
**trade-off/uncertain**, or **no demonstrated advantage** -- produced purely from the frozen rule
imported from the primary-evaluation workflow, not a written conclusion.


In [ ]:
classification_by_cell = complexity_value_summary.drop_duplicates(
    ["endpoint", "design", "centre", "fuller_model"]
)[["endpoint", "design", "centre", "fuller_model", "classification"]]
display(classification_by_cell["classification"].value_counts())
display(classification_by_cell)

classification
no_demonstrated_advantage    35
trade_off_uncertain           8
clear_added_value             2
Name: count, dtype: int64

,endpoint,design,centre,fuller_model,classification
0,OS,iecv,DFCI,glmnet_ridge_core,no_demonstrated_advantage
7,OS,iecv,DFCI,glmnet_enet_core,no_demonstrated_advantage
14,OS,iecv,DFCI,penalized_nonlinear_cox,no_demonstrated_advantage
21,OS,iecv,DFCI,reduced_core_penalized_cox,no_demonstrated_advantage
28,OS,iecv,DFCI,flexsurv_rp_spline,trade_off_uncertain
35,OS,pooled,DFCI,glmnet_ridge_core,no_demonstrated_advantage
42,OS,pooled,DFCI,glmnet_enet_core,no_demonstrated_advantage
49,OS,pooled,DFCI,penalized_nonlinear_cox,no_demonstrated_advantage
56,OS,pooled,DFCI,reduced_core_penalized_cox,no_demonstrated_advantage
63,OS,pooled,DFCI,flexsurv_rp_spline,trade_off_uncertain


### Success criteria, evidence by criterion

`optimization-spec.md`'s "Success criteria" requires all seven to hold for a credible improvement,
and explicitly prohibits constructing a composite score. Criteria 1-2 (material primary-target IBS
improvement with compatible paired uncertainty) are directly mechanical from the `ibs` gain already
computed above with its locked materiality margin (`0.005`), restricted to the **IECV** design --
the institution-held-out test, which is the more conservative primary comparison; the pooled-design
numbers are shown separately in "Transportability gap" above rather than mixed into this table.
Criteria 3-4 (calibration, no important worsening) and 6-7 (cross-centre consistency, no
single-comparison dependence) require judgment beyond one number by design -- the table below
reports the underlying evidence for a reader to apply them, rather than fabricating a single
pass/fail flag the specification says must not exist.


In [ ]:
IBS_MATERIALITY_MARGIN = 0.005

os_ibs_gain = complexity_value_summary[
    (complexity_value_summary["endpoint"] == "OS")
    & (complexity_value_summary["design"] == "iecv")
    & (complexity_value_summary["gain"] == "ibs")
][["centre", "fuller_model", "value", "ci_lower", "ci_upper"]].rename(
    columns={"value": "os_ibs_gain"}
)
os_ibs_gain["criterion_1_2_material_and_compatible"] = (
    os_ibs_gain["os_ibs_gain"] >= IBS_MATERIALITY_MARGIN
) & (os_ibs_gain["ci_lower"] > 0)
display(os_ibs_gain.sort_values(["fuller_model", "centre"]))

fit_success_rate = (
    development_result.fit_summary.groupby("candidate")["fit_status"]
    .apply(lambda s: (s == "fit_succeeded").mean())
    .rename("criterion_5_fit_success_rate")
)
display(fit_success_rate)

os_classification_by_centre = classification_by_cell[
    (classification_by_cell["endpoint"] == "OS") & (classification_by_cell["design"] == "iecv")
][["centre", "fuller_model", "classification"]].pivot(
    index="fuller_model", columns="centre", values="classification"
)
display(os_classification_by_centre)

,centre,fuller_model,os_ibs_gain,ci_lower,ci_upper,criterion_1_2_material_and_compatible
30,DFCI,flexsurv_rp_spline,NaN,NaN,NaN,False
100,MSK,flexsurv_rp_spline,-0.017538,-0.032371,-0.002436,False
170,VICC,flexsurv_rp_spline,NaN,NaN,NaN,False
9,DFCI,glmnet_enet_core,0.000077,-0.014929,0.015759,False
79,MSK,glmnet_enet_core,-0.012534,-0.024250,-0.000419,False
149,VICC,glmnet_enet_core,0.012565,-0.014643,0.040543,False
2,DFCI,glmnet_ridge_core,0.002132,-0.011344,0.016583,False
72,MSK,glmnet_ridge_core,-0.006040,-0.015554,0.003914,False
142,VICC,glmnet_ridge_core,0.013389,-0.009722,0.040059,False
16,DFCI,penalized_nonlinear_cox,0.002130,-0.004400,0.009343,False


candidate
flexsurv_rp_spline            0.428571
glmnet_enet_core              1.000000
glmnet_ridge_core             1.000000
penalized_nonlinear_cox       1.000000
reduced_core_penalized_cox    1.000000
Name: criterion_5_fit_success_rate, dtype: float64

centre,DFCI,MSK,VICC
fuller_model,,,
flexsurv_rp_spline,trade_off_uncertain,no_demonstrated_advantage,trade_off_uncertain
glmnet_enet_core,no_demonstrated_advantage,no_demonstrated_advantage,no_demonstrated_advantage
glmnet_ridge_core,no_demonstrated_advantage,no_demonstrated_advantage,no_demonstrated_advantage
penalized_nonlinear_cox,no_demonstrated_advantage,no_demonstrated_advantage,no_demonstrated_advantage
reduced_core_penalized_cox,no_demonstrated_advantage,no_demonstrated_advantage,no_demonstrated_advantage


**Statement of which success criteria each candidate met:** zero `B2`-`B6` candidates show a
material, uncertainty-compatible `ibs_gain` for the primary 730-day OS target at any centre
(criterion 1-2 fails for all five, at every centre where computable) -- the `os_ibs_gain` table
above has no `True` value in `criterion_1_2_material_and_compatible`. Because criterion 1 is
required and not met by any candidate, none of `B2`-`B6` meets the full seven-criterion bar for a
credible improvement on the primary target. Under the claim limits in `optimization-spec.md`, this
is reported as the finding itself: **a prospectively specified exploratory
optimization did not improve on the minimal Cox benchmark for 730-day OS**, which strengthens
rather than weakens the frozen primary-evaluation conclusion that added complexity showed no clear value for
OS. The only `clear_added_value` cells anywhere in the table above belong to the supportive TTNT-D
endpoint at a single centre, which `optimization-spec.md` prospectively excludes from the primary
conclusion.


## Recalibration

Not performed this pass. `optimization-spec.md` makes intercept-only/intercept-plus-slope local
adaptation recalibration optional and permits deferring it to the sensitivity-analysis workflow, which owns calibration drift
and the local-adaptation learning curve -- recorded here with an explicit deferral status, not omitted
silently.


## Canonical artifacts and privacy scan

Workstream A publishes to a dedicated correction directory; Workstream B publishes to a dedicated
exploratory-optimization directory. Both resolve under `analysis/out/`, matched by the
project's blanket `.gitignore` entry, so neither can be staged. The frozen model-training and primary-evaluation bundle is
re-verified byte-for-byte unchanged one last time below, after every real call this notebook made.


In [ ]:
import re

verification_after = verify_frozen_primary_unchanged(ANALYSIS_DIRECTORY, FEATURE_MANIFEST_PATH)
assert verification_after == verification_before, (
    "Frozen model-training and primary-evaluation bundle changed during this notebook's execution"
)
display(verification_after)

privacy_pattern = re.compile(r"GENIE-[A-Z0-9-]+")
scan_targets = list(WORKSTREAM_B_DIRECTORY.glob("*.csv")) + list(
    WORKSTREAM_B_DIRECTORY.glob("*.json")
)
patient_id_hits = sum(
    len(privacy_pattern.findall(path.read_text(encoding="utf-8"))) for path in scan_targets
)
display({"published_files_scanned": len(scan_targets), "patient_id_pattern_hits": patient_id_hits})
assert patient_id_hits == 0, "Patient-identifier pattern found in a published Workstream B artifact"

display(
    pd.Series(
        {
            "workstream_a_output": str(correction_result.output_directory),
            "workstream_b_output": str(WORKSTREAM_B_DIRECTORY),
            "workstream_b_fit_summary": "fit_summary.csv",
            "workstream_b_model_registry": "model_registry.json",
            "workstream_b_r_model_registry": "r_model_registry.json",
            "workstream_b_performance_summary": "performance_summary.csv",
            "workstream_b_transportability_summary": "transportability_summary.csv",
            "workstream_b_complexity_value_summary": "complexity_value_summary.csv",
        }
    )
)

FrozenPrimaryVerification(cleaning_run_manifest_sha256='d11982f0e9cd7f740cae7432744fa23aba336d944f7236a2f10429e775b84bfd', analysis_dataset_sha256='6b9e28099f547150324b33be900d228d22b411b7dbe0d68e1e08551aaab6740e', feature_manifest_sha256='cc891bcdca2b06179dd33e4bc55160e0e048557059e57b0b02789c7b18dc0f73', model_training_fit_summary_sha256='dfb77a84220741ad94e1362a90eaaa8c2240980ddd7c59ef0ed039548d60ad9b', model_training_registry_sha256='2695e55ae3ed97a9e98d5ff419544dafb7ee21e9bb65c6e6ca4ae6bd3770596f', primary_evaluation_manifest_sha256='1e25224a40ce91b4f88d16410f658291f2440e83209e4330a70d274a2860ba15')

{'published_files_scanned': 7, 'patient_id_pattern_hits': 0}

workstream_a_output                      /Users/camilasarmiento/Documents/Data Freelanc...
workstream_b_output                      /Users/camilasarmiento/Documents/Data Freelanc...
workstream_b_fit_summary                                                   fit_summary.csv
workstream_b_model_registry                                            model_registry.json
workstream_b_r_model_registry                                        r_model_registry.json
workstream_b_performance_summary                                   performance_summary.csv
workstream_b_transportability_summary                         transportability_summary.csv
workstream_b_complexity_value_summary                         complexity_value_summary.csv
dtype: object